In [14]:
from pathlib import Path
import os
print("Lo lagi di:", Path.cwd())
print("--- isi folder ---")
for p in Path.cwd().iterdir():
    print(p.name)

# Cari shipments.csv dimana aja
print("--- nyari shipments.csv ---")
for p in Path.cwd().rglob("shipments.csv"):
    print(p)

Lo lagi di: c:\Users\Pitri Wulandari\Documents\Datavidia\pitri\notebooks
--- isi folder ---
.gitkeep
baseline_lgbm.ipynb
train_v2.ipynb
--- nyari shipments.csv ---


In [15]:
from pathlib import Path
# naik 2 level ke folder Datavidia
base = Path.cwd().parent.parent
print("Cek di:", base)
for p in base.iterdir():
    print(p.name)

print("--- nyari dari Datavidia ---")
for p in base.rglob("shipments.csv"):
    print(p)
    break

Cek di: c:\Users\Pitri Wulandari\Documents\Datavidia
.git
.gitignore
data
docs
farhat
ghani
notebooks
pitri
README.md
requirements.txt
src
submissions
--- nyari dari Datavidia ---
c:\Users\Pitri Wulandari\Documents\Datavidia\data\raw\task-1-datavidia-11-0\Task-1-Datavidia-11.0\test\shipments.csv


In [16]:
import pandas as pd, numpy as np
from pathlib import Path

# Path yang bener buat lo
DATA_ROOT = Path.cwd().parent.parent / "data" / "raw" / "task-1-datavidia-11-0" / "Task-1-Datavidia-11.0"
ROOT = DATA_ROOT / "train"
print("ROOT:", ROOT)
print("Ada file?", (ROOT/"shipments.csv").exists())

s = pd.read_csv(ROOT/"shipments.csv")
print("shipments:", s.shape)

# --- scan_events ---
scan = pd.read_csv(ROOT/"scan_events.csv", usecols=["shipment_id","event_type","gap_hours","scan_quality","is_manual"])
g = scan.groupby("shipment_id")
scan_agg = g.agg(scan_n=("event_type","count"), scan_gap_mean=("gap_hours","mean"), scan_gap_max=("gap_hours","max"), scan_q_mean=("scan_quality","mean"), scan_manual_mean=("is_manual","mean"))
cnt = pd.crosstab(scan["shipment_id"], scan["event_type"]).add_prefix("scan_cnt_")
s = s.merge(scan_agg, left_on="shipment_id", right_index=True, how="left").merge(cnt, left_on="shipment_id", right_index=True, how="left")
print("habis scan:", s.shape)

# --- route_segments ---
route = pd.read_csv(ROOT/"route_segments.csv", usecols=["shipment_id","jarak_segmen_km","durasi_jam","biaya_segmen","moda"])
g2 = route.groupby("shipment_id")
route_agg = g2.agg(seg_n=("jarak_segmen_km","count"), seg_dist_sum=("jarak_segmen_km","sum"), seg_dur_sum=("durasi_jam","sum"), seg_cost_sum=("biaya_segmen","sum"))
s = s.merge(route_agg, left_on="shipment_id", right_index=True, how="left")
s["seg_speed"] = s["seg_dist_sum"] / (s["seg_dur_sum"] + 1e-6)
print("habis route:", s.shape)

# --- customers, couriers, hubs, packages ---
for file, key, prefix in [("customers.csv","customer_id","cu_"),("couriers.csv","courier_id","co_"),("hubs.csv","hub_id","hu_"),("service_packages.csv","package_id","sp_")]:
    t = pd.read_csv(ROOT/file).add_prefix(prefix).rename(columns={prefix+key: key})
    s = s.merge(t, on=key, how="left")
print("final:", s.shape)

# --- fitur buatan ---
s["vol_ratio"] = s["berat_volumetrik"] / (s["weight_kg"] + 1e-6)
s["value_per_kg"] = s["declared_value"] / (s["weight_kg"] + 1e-6)
s["est_vs_dur"] = s["seg_dur_sum"] / (s["sp_estimasi_hari"]*24 + 1e-6)

df = s.copy()
df.to_pickle("df_pitri.pkl")
print("KESIMPEN df_pitri.pkl")

ROOT: c:\Users\Pitri Wulandari\Documents\Datavidia\data\raw\task-1-datavidia-11-0\Task-1-Datavidia-11.0\train
Ada file? True
shipments: (1243617, 38)


MemoryError: Unable to allocate 35.8 MiB for an array with shape (4688502,) and data type int64

In [5]:
from pathlib import Path
OUT = ROOT / "pitri" / "data"
OUT.mkdir(parents=True, exist_ok=True)
print("Simpan ke:", OUT)

# 1. SCAN
print("Bikin scan_feat...")
scan = pd.read_csv(BASE/"scan_events.csv")
scan_feat = scan.groupby('shipment_id').agg(
    scan_count=('event_id','count'),
    scan_gap_mean=('gap_hours','mean'),
    scan_gap_max=('gap_hours','max'),
    scan_manual_rate=('is_manual','mean'),
    scan_quality_mean=('scan_quality','mean'),
    scan_unique_hub=('hub_id','nunique'),
    scan_unique_event=('event_type','nunique'),
).reset_index()
scan_feat.to_csv(OUT/"scan_feat.csv", index=False)
del scan, scan_feat; gc.collect()
print("scan_feat OK")

# 2. INVOICE
print("Bikin inv_feat...")
inv = pd.read_csv(BASE/"invoices.csv")
inv_feat = inv.groupby('shipment_id').agg(
    inv_count=('invoice_id','count'),
    inv_amount_sum=('amount','sum'),
    inv_amount_mean=('amount','mean'),
    inv_discount_sum=('discount','sum'),
    inv_tax_sum=('tax','sum'),
    inv_days_to_pay_mean=('days_to_pay','mean'),
    inv_penalty_rate=('is_penalty','mean'),
).reset_index()
inv_feat.to_csv(OUT/"inv_feat.csv", index=False)
del inv, inv_feat; gc.collect()
print("inv_feat OK")

# 3. ROUTE
print("Bikin route_feat...")
route = pd.read_csv(BASE/"route_segments.csv")
route_feat = route.groupby('shipment_id').agg(
    route_n_segments=('segment_id','count'),
    route_jarak_total=('jarak_segmen_km','sum'),
    route_durasi_total=('durasi_jam','sum'),
    route_biaya_total=('biaya_segmen','sum'),
    route_jarak_mean=('jarak_segmen_km','mean'),
    route_durasi_mean=('durasi_jam','mean'),
    route_unique_moda=('moda','nunique'),
).reset_index()
route_feat.to_csv(OUT/"route_feat.csv", index=False)
del route, route_feat; gc.collect()
print("SEMUA BERES!")

Simpan ke: c:\Users\Pitri Wulandari\Documents\Datavidia\pitri\data
Bikin scan_feat...
scan_feat OK
Bikin inv_feat...
inv_feat OK
Bikin route_feat...
SEMUA BERES!


In [6]:
import pandas as pd, numpy as np, gc
from pathlib import Path

ROOT = Path.cwd().parents[1] if "pitri" in str(Path.cwd()) else Path.cwd()
BASE = ROOT / "data/raw/task-1-datavidia-11-0/Task-1-Datavidia-11.0/train"
OUT = ROOT / "pitri" / "data"

def find_file(name):
    for p in [ROOT/"data", ROOT/"pitri/data", ROOT/"ghani/data", ROOT/"pitri", ROOT/"ghani"]:
        f = p / name
        if f.exists():
            print(f"Ketemu {name} di {f}")
            return f
    raise FileNotFoundError(f"{name} ga ketemu!")

ship = pd.read_csv(BASE / "shipments.csv", nrows=200000)
print("ship:", ship.shape)
print(ship.columns.tolist())
ship.head()

ship: (200000, 38)
['shipment_id', 'customer_id', 'courier_id', 'hub_id', 'package_id', 'promo_id', 'weight_kg', 'volume_l', 'declared_value', 'distance_km', 'ship_month', 'ship_dow', 'ship_hour', 'is_cod', 'is_promo', 'channel', 'pickup_type', 'goods_category', 'dest_region', 'insurance_opt', 'qty_items', 'coupon_count', 'gift_wrap', 'fragility_flag', 'order_to_ship_hours', 'prioritas_pengiriman', 'jumlah_catatan', 'flag_alamat_baru', 'berat_volumetrik', 'kota_asal', 'kota_tujuan', 'skor_prioritas_kirim', 'indeks_rute_sekunder', 'faktor_muatan_internal', 'varian_cuaca_rute', 'indeks_waktu_proses', 'skor_kualitas_cek', 'status_penanganan']


,shipment_id,customer_id,courier_id,hub_id,package_id,promo_id,weight_kg,volume_l,declared_value,distance_km,...,berat_volumetrik,kota_asal,kota_tujuan,skor_prioritas_kirim,indeks_rute_sekunder,faktor_muatan_internal,varian_cuaca_rute,indeks_waktu_proses,skor_kualitas_cek,status_penanganan
0,SHP-AHSVFB3R,CUS-5VB7S4ZY,CRV-DZEF3GM9,HUB-MSVB47XA,PKG-NZ3YX33Z,TANPA_PROMO,3.75,7.9,16000,1277.6,...,6.69,banyuwangi,jayapura,39.50,43.72,37.79,47.06,57.00,55.10,normal
1,SHP-FUWWVBCQ,CUS-79DW3KY8,CRV-S8LG8JQP,HUB-B83PJTMD,PKG-GTVK7E6E,PRM-F82RASMM,1.46,3.4,23000,515.9,...,3.56,manado,palu,48.35,63.08,44.36,63.36,35.57,46.20,retur_pengirim
2,SHP-9EBN7MB9,CUS-642HAPWJ,CRV-EQ3SX46G,HUB-AGDQJQVF,PKG-QPNPLAVN,TANPA_PROMO,3.30,8.0,119000,653.1,...,7.18,bandung,solo,59.01,55.77,55.04,54.16,48.49,36.90,normal
3,SHP-5LK9F3MX,CUS-QDZSXZWD,CRV-XYHRH2DD,HUB-22X96KLN,PKG-2Y2RN6AW,TANPA_PROMO,4.77,7.5,98000,434.5,...,12.34,surabaya,pekanbaru,47.49,49.37,61.23,42.10,50.49,60.30,retur_pengirim
4,SHP-6XV4H3CP,CUS-QADQDZQ3,CRV-LQWU6RFS,HUB-YZUUKVTW,PKG-R3P3KWCQ,PRM-EAXBNEFE,3.24,6.4,86000,997.4,...,3.98,kudus,purwokerto,47.34,56.25,54.06,52.92,44.46,45.72,konfirmasi_alamat


In [7]:
# Kalau scan_feat.csv udah ada, SKIP cell ini
import os
if not (OUT/"scan_feat.csv").exists():
    OUT.mkdir(parents=True, exist_ok=True)
    scan = pd.read_csv(BASE/"scan_events.csv")
    scan.groupby('shipment_id').agg(scan_count=('event_id','count'), scan_gap_mean=('gap_hours','mean'), scan_gap_max=('gap_hours','max'), scan_manual_rate=('is_manual','mean'), scan_quality_mean=('scan_quality','mean'), scan_unique_hub=('hub_id','nunique'), scan_unique_event=('event_type','nunique')).reset_index().to_csv(OUT/"scan_feat.csv", index=False)
    del scan; gc.collect()
    inv = pd.read_csv(BASE/"invoices.csv")
    inv.groupby('shipment_id').agg(inv_count=('invoice_id','count'), inv_amount_sum=('amount','sum'), inv_amount_mean=('amount','mean'), inv_discount_sum=('discount','sum'), inv_tax_sum=('tax','sum'), inv_days_to_pay_mean=('days_to_pay','mean'), inv_penalty_rate=('is_penalty','mean')).reset_index().to_csv(OUT/"inv_feat.csv", index=False)
    del inv; gc.collect()
    route = pd.read_csv(BASE/"route_segments.csv")
    route.groupby('shipment_id').agg(route_n_segments=('segment_id','count'), route_jarak_total=('jarak_segmen_km','sum'), route_durasi_total=('durasi_jam','sum'), route_biaya_total=('biaya_segmen','sum'), route_jarak_mean=('jarak_segmen_km','mean'), route_durasi_mean=('durasi_jam','mean'), route_unique_moda=('moda','nunique')).reset_index().to_csv(OUT/"route_feat.csv", index=False)
    del route; gc.collect()
    print("FEAT DIBIKIN")
else:
    print("FEAT UDAH ADA, SKIP")

FEAT UDAH ADA, SKIP


In [8]:
scan = pd.read_csv(find_file("scan_feat.csv"))
inv = pd.read_csv(find_file("inv_feat.csv"))
route = pd.read_csv(find_file("route_feat.csv"))
df = ship.merge(scan, on='shipment_id', how='left').merge(inv, on='shipment_id', how='left').merge(route, on='shipment_id', how='left')
del ship, scan, inv, route; gc.collect()

cour = pd.read_csv(BASE / "couriers.csv").drop(columns=['hub_id'])
cust = pd.read_csv(BASE / "customers.csv")
hub = pd.read_csv(BASE / "hubs.csv").add_prefix('hub_').rename(columns={'hub_hub_id':'hub_id'})
pkg = pd.read_csv(BASE / "service_packages.csv").add_prefix('pkg_').rename(columns={'pkg_package_id':'package_id'})
promo = pd.read_csv(BASE / "promotions.csv")[['promo_id','diskon_persen','tipe_promo']]

df = df.merge(cour, on='courier_id', how='left').merge(cust, on='customer_id', how='left')
df = df.merge(hub, on='hub_id', how='left').merge(pkg, on='package_id', how='left').merge(promo, on='promo_id', how='left')
del cour, cust, hub, pkg, promo; gc.collect()
print(df.shape)
df.head()

Ketemu scan_feat.csv di c:\Users\Pitri Wulandari\Documents\Datavidia\pitri\data\scan_feat.csv
Ketemu inv_feat.csv di c:\Users\Pitri Wulandari\Documents\Datavidia\pitri\data\inv_feat.csv
Ketemu route_feat.csv di c:\Users\Pitri Wulandari\Documents\Datavidia\pitri\data\route_feat.csv
(200000, 100)


,shipment_id,customer_id,courier_id,hub_id,package_id,promo_id,weight_kg,volume_l,declared_value,distance_km,...,pkg_handling_class,pkg_insurance_default,pkg_cod_allowed,pkg_cost_multiplier,pkg_max_weight_kg,pkg_fragile_score,pkg_estimasi_hari,pkg_biaya_dasar,diskon_persen,tipe_promo
0,SHP-AHSVFB3R,CUS-5VB7S4ZY,CRV-DZEF3GM9,HUB-MSVB47XA,PKG-NZ3YX33Z,TANPA_PROMO,3.75,7.9,16000,1277.6,...,standar,1,0,1.14,5,0.30,2,43000,NaN,NaN
1,SHP-FUWWVBCQ,CUS-79DW3KY8,CRV-S8LG8JQP,HUB-B83PJTMD,PKG-GTVK7E6E,PRM-F82RASMM,1.46,3.4,23000,515.9,...,standar,0,0,1.14,2,0.07,1,20000,21.4,gratis_ongkir
2,SHP-9EBN7MB9,CUS-642HAPWJ,CRV-EQ3SX46G,HUB-AGDQJQVF,PKG-QPNPLAVN,TANPA_PROMO,3.30,8.0,119000,653.1,...,standar,1,1,0.80,20,0.11,5,13000,NaN,NaN
3,SHP-5LK9F3MX,CUS-QDZSXZWD,CRV-XYHRH2DD,HUB-22X96KLN,PKG-2Y2RN6AW,TANPA_PROMO,4.77,7.5,98000,434.5,...,fragile,0,0,0.86,40,0.14,1,13000,NaN,NaN
4,SHP-6XV4H3CP,CUS-QADQDZQ3,CRV-LQWU6RFS,HUB-YZUUKVTW,PKG-R3P3KWCQ,PRM-EAXBNEFE,3.24,6.4,86000,997.4,...,fragile,1,0,1.31,2,0.54,1,19000,21.5,gratis_ongkir


In [12]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import lightgbm as lgb

TARGET = 'status_penanganan'
print("TARGET:", TARGET)
print(df[TARGET].value_counts())

df = df.dropna(subset=[TARGET]).copy()

# Drop ID + tanggal, SAMA kayak Ghani
drop_cols = ['shipment_id','customer_id','courier_id','package_id','promo_id','hub_id', TARGET]
for c in list(df.columns):
    if any(k in c.lower() for k in ['date','time','tanggal','created','pickup','delivery','estimasi']):
        if c not in drop_cols:
            drop_cols.append(c)

feat_cols = [c for c in df.columns if c not in drop_cols]

for c in feat_cols:
    if pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].fillna(-1)
    else:
        df[c] = df[c].fillna('Unknown').astype(str).astype('category')

# Encode target jadi 0,1,2...
classes = sorted(df[TARGET].unique())
print("Classes:", classes)
y = df[TARGET].map({c:i for i,c in enumerate(classes)}).values
X = df[feat_cols]

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)

model = lgb.LGBMClassifier(objective='multiclass', num_class=len(classes), n_estimators=400, learning_rate=0.1, num_leaves=63, min_child_samples=100, subsample=0.8, colsample_bytree=0.5, reg_lambda=10, random_state=42, verbose=-1)
model.fit(X_train, y_train)

pred = model.predict(X_val)
print("Macro F1:", f1_score(y_val, pred, average='macro'))
print(classification_report(y_val, pred, target_names=classes))

imp = pd.DataFrame({'feature':feat_cols, 'importance':model.feature_importances_}).sort_values('importance', ascending=False)
print(imp.head(20))

import joblib
Path(ROOT/"pitri/models").mkdir(parents=True, exist_ok=True)
joblib.dump(model, ROOT/"pitri/models/train_v2.pkl")
joblib.dump(classes, ROOT/"pitri/models/classes.pkl")
print("MODEL KESIMPEN!")

TARGET: status_penanganan
status_penanganan
normal                  12805
konfirmasi_alamat        7421
tertunda_operasional     5135
gagal_antar_awal         3364
retur_pengirim           2370
rusak_perjalanan         1684
klaim_hilang             1038
Name: count, dtype: int64
Classes: ['gagal_antar_awal', 'klaim_hilang', 'konfirmasi_alamat', 'normal', 'retur_pengirim', 'rusak_perjalanan', 'tertunda_operasional']
Macro F1: 0.41414771747793855
                      precision    recall  f1-score   support

    gagal_antar_awal       0.37      0.22      0.27       505
        klaim_hilang       0.47      0.18      0.26       156
   konfirmasi_alamat       0.58      0.66      0.62      1113
              normal       0.65      0.82      0.73      1921
      retur_pengirim       0.32      0.14      0.20       355
    rusak_perjalanan       0.44      0.22      0.30       253
tertunda_operasional       0.54      0.52      0.53       770

            accuracy                           0.58  

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
import lightgbm as lgb

ROOT = Path(r"c:\Users\Pitri Wulandari\Documents\Datavidia\data\raw\task-1-datavidia-11-0\Task-1-Datavidia-11.0\train")
print("ROOT ada:", ROOT.exists())
print([f.name for f in sorted(ROOT.glob("*.csv"))])

s = pd.read_csv(ROOT / "shipments.csv")
print("shipments columns:", s.columns.tolist())
print("shape:", s.shape)

target_col = None
for c in s.columns:
    if "status" in c.lower() or "penanganan" in c.lower() or "label" in c.lower():
        target_col = c
        break
print("TARGET ketemu:", target_col)
print(s[target_col].value_counts(dropna=False))

df = s.copy()

if (ROOT / "scan_events.csv").exists():
    sc = pd.read_csv(ROOT / "scan_events.csv")
    print("scan_events columns:", sc.columns.tolist())
    gap_col = None
    for c in sc.columns:
        if "gap" in c.lower() or "selisih" in c.lower() or "interval" in c.lower() or "hours" in c.lower():
            gap_col = c
            break
    if gap_col is not None and "shipment_id" in sc.columns:
        sc_agg = sc.groupby("shipment_id")[gap_col].agg(scan_n="size", scan_gap_max="max", scan_q_mean="mean").reset_index()
        df = df.merge(sc_agg, on="shipment_id", how="left")
    elif "shipment_id" in sc.columns:
        sc_agg = sc.groupby("shipment_id").size().reset_index(name="scan_n")
        df = df.merge(sc_agg, on="shipment_id", how="left")

if (ROOT / "route_segments.csv").exists():
    rt = pd.read_csv(ROOT / "route_segments.csv")
    print("route_segments columns:", rt.columns.tolist())
    agg_dict = {}
    for c in rt.columns:
        cl = c.lower()
        if "duration" in cl or "durasi" in cl or "time" in cl or "menit" in cl:
            agg_dict[c] = "sum"
        if "distance" in cl or "jarak" in cl or "km" in cl:
            agg_dict[c] = "sum"
        if "cost" in cl or "biaya" in cl or "idr" in cl or "harga" in cl:
            agg_dict[c] = "sum"
    if agg_dict and "shipment_id" in rt.columns:
        rt_agg = rt.groupby("shipment_id").agg(agg_dict).reset_index()
        rename_map = {}
        for c in rt_agg.columns:
            cl = c.lower()
            if "duration" in cl or "durasi" in cl or "time" in cl:
                rename_map[c] = "seg_durasi_total"
            if "distance" in cl or "jarak" in cl:
                rename_map[c] = "seg_dist_sum"
            if "cost" in cl or "biaya" in cl or "idr" in cl:
                rename_map[c] = "seg_cost_sum"
        rt_agg = rt_agg.rename(columns=rename_map)
        df = df.merge(rt_agg, on="shipment_id", how="left")

if "berat_volumetrik" in df.columns and "weight_kg" in df.columns:
    df["vol_ratio"] = df["berat_volumetrik"] / (df["weight_kg"] + 0.1)
elif "volume_l" in df.columns and "weight_kg" in df.columns:
    df["vol_ratio"] = df["volume_l"] / (df["weight_kg"] + 0.1)

if "scan_gap_max" in df.columns:
    df["is_scan_jebol"] = (df["scan_gap_max"] > 24).astype(int)

if "seg_cost_sum" in df.columns and "seg_dist_sum" in df.columns:
    df["cost_per_km"] = df["seg_cost_sum"] / (df["seg_dist_sum"] + 1)

if "vol_ratio" in df.columns:
    df["is_berat_ganjil"] = (df["vol_ratio"] > 3).astype(int)

print("df final shape:", df.shape)

drop_cols = ["shipment_id", target_col, "customer_id", "courier_id", "hub_id", "package_id", "promo_id", "kota_asal", "kota_tujuan"]
X = df.drop(columns=[c for c in drop_cols if c in df.columns])
cat_cols = X.select_dtypes(include=["object", "string", "category"]).columns.tolist()
X = pd.get_dummies(X, columns=cat_cols, drop_first=True)
X = X.apply(pd.to_numeric, errors="coerce").fillna(0)

le = LabelEncoder()
y = le.fit_transform(df[target_col].astype(str))

params = {
    "objective": "multiclass",
    "num_class": len(le.classes_),
    "metric": "multi_logloss",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "feature_fraction": 0.8,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "class_weight": "balanced",
    "verbose": -1,
    "n_jobs": -1,
    "random_state": 42
}

skf = StratifiedKFold(5, shuffle=True, random_state=42)
oof = np.zeros((len(X), len(le.classes_)))
for fold, (tr, va) in enumerate(skf.split(X, y)):
    m = lgb.LGBMClassifier(**params, n_estimators=800)
    m.fit(X.iloc[tr], y[tr], eval_set=[(X.iloc[va], y[va])], callbacks=[lgb.early_stopping(100, verbose=False)])
    oof[va] = m.predict_proba(X.iloc[va])
    print(f"Fold {fold} done")

pred = oof.argmax(1)
print("Macro F1 OOF:", f1_score(y, pred, average="macro"))
print(classification_report(y, pred, target_names=le.classes_))

ROOT ada: True
['couriers.csv', 'customers.csv', 'hubs.csv', 'invoices.csv', 'promotions.csv', 'route_segments.csv', 'scan_events.csv', 'service_packages.csv', 'shipments.csv']
shipments columns: ['shipment_id', 'customer_id', 'courier_id', 'hub_id', 'package_id', 'promo_id', 'weight_kg', 'volume_l', 'declared_value', 'distance_km', 'ship_month', 'ship_dow', 'ship_hour', 'is_cod', 'is_promo', 'channel', 'pickup_type', 'goods_category', 'dest_region', 'insurance_opt', 'qty_items', 'coupon_count', 'gift_wrap', 'fragility_flag', 'order_to_ship_hours', 'prioritas_pengiriman', 'jumlah_catatan', 'flag_alamat_baru', 'berat_volumetrik', 'kota_asal', 'kota_tujuan', 'skor_prioritas_kirim', 'indeks_rute_sekunder', 'faktor_muatan_internal', 'varian_cuaca_rute', 'indeks_waktu_proses', 'skor_kualitas_cek', 'status_penanganan']
shape: (1243617, 38)
TARGET ketemu: status_penanganan
status_penanganan
normal                  473531
konfirmasi_alamat       272911
tertunda_operasional    185496
gagal_anta